In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from app.segmentation.downloader import LidcIdriDownloader
from app.segmentation.patient_manager import PatientManager

## Get patients from BDD LIBDC-IDRI

In [ ]:
patient_ids = [f'LIDC-IDRI-{i:04d}' for i in range(1, 10)]

dl = LidcIdriDownloader("./LIDC_data/", patient_ids)
dl.fill_patients_files_info()

In [ ]:
dl.download()

## Work on patients

In [ ]:
patient0 = PatientManager(patient_ids[7], dl)

In [ ]:
patient0.init()

In [ ]:
patient0.display_volume_with_annotations()

## Segmentation

In [ ]:
from app.segmentation.segmenter import Segmenter
seg = Segmenter(patient0)
candidates = seg.run()

In [ ]:
slice_idx = 76
seg.display_lung_mask(slice_idx)
seg.display_roi(slice_idx)
seg.display_segmentation(slice_idx)

### Tests

In [ ]:
ann_candidates = patient0.get_annotation_candidates()

In [ ]:
annotations_mask = patient0.get_volume_with_annotations()
ann_candidates = patient0.get_annotation_candidates()

pairs = Segmenter.match_candidates(ann_candidates, seg.candidates)
iou_scores = [
    Segmenter.compute_iou_3d(ann, cand, annotations_mask, seg.nodules_mask)
    for ann, cand in pairs
]

In [ ]:
pairs

In [ ]:
iou_scores

In [ ]:
recall = len(pairs) / len(ann_candidates)
print(f"Recall: {recall:.2%} - {len(pairs)}/{len(ann_candidates)} annotations matched")

In [ ]:
z, y, x = ann_candidates[1]["centroid"]

In [ ]:
print(patient0.volume[z, y-5:y+5, x-5:x+5])